# 15.03 · Distillation & synthetic data

**Chapter:** Advanced & Frontier Topics · **Milestone:** M7

## Goals
- [ ] Build a synthetic data pipeline: **seed → generate → deduplicate → filter → split**, and measure diversity
- [ ] Distill a large "teacher" model's behaviour on one task into a tiny, fast "student"
- [ ] Measure what you gained (latency, cost) and lost (accuracy) on a **hand-labelled** test set the teacher never generated
- [ ] Know the variants: hard-label vs soft-label (logit) distillation, on-policy distillation, self-distillation, rejection sampling
- [ ] Know the **licensing / terms-of-service** constraints on training with model outputs

## Resources
- [HF – Synthetic data generator](https://huggingface.co/blog/synthetic-data-generator) · [distilabel](https://distilabel.argilla.io/) — free
- [Hinton et al. – Distilling the knowledge in a neural network](https://arxiv.org/abs/1503.02531) — free
- [Self-Instruct](https://arxiv.org/abs/2212.10560) · [Textbooks Are All You Need (phi-1)](https://arxiv.org/abs/2306.11644) · [Cosmopedia](https://huggingface.co/blog/cosmopedia) — free
- [Thinking Machines – On-policy distillation](https://thinkingmachines.ai/blog/on-policy-distillation/) — free

The task: route customer messages of an online store (in English and Spanish) to one of six queues. The teacher is a big LLM (accurate,
~1 s, costs per call). The student will be an embedding + logistic-regression classifier (~1 ms, free to run) trained **only on data the
teacher wrote and labelled**.

In [ ]:
import itertools
import json
import random
import re
import time

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from aieng import DATA, rag
from aieng.llm import available, client

TEACHER = "groq" if "groq" in available() else "gemini" if "gemini" in available() else "ollama"
tc, TMODEL = client(TEACHER)
EXTRA = {"reasoning_effort": "low"} if TEACHER == "groq" else {"reasoning_effort": "none"} if TEACHER == "ollama" else {}
PAUSE = {"groq": 2, "gemini": 4}.get(TEACHER, 0)  # free-tier rate limits
print("teacher:", TEACHER, TMODEL)

INTENTS = {
    "billing": "charges, invoices, refunds of money, payment methods, double charges",
    "shipping": "delivery status, delays, tracking, address changes before delivery",
    "returns": "returning or exchanging a product, return labels, damaged or wrong items",
    "account": "login, password, email change, deleting the account, personal data",
    "product": "questions about a product before buying: sizes, specs, stock, compatibility",
    "other": "anything else: feedback, partnerships, jobs, spam, greetings",
}

## 1. The gold test set: written by a human, never by the teacher
Everything else in this notebook is synthetic, so the test set must not be, or we'd only measure how well the student imitates the
teacher's style. 48 messages, 8 per intent, messy on purpose (typos, mixed languages, ambiguous cases).

In [ ]:
GOLD = [
    ("I was charged twice for order 4471, pls fix", "billing"), ("me cobraron dos veces la misma compra", "billing"),
    ("Can I pay with PayPal or only card?", "billing"), ("where is my refund?? returned the shoes 3 weeks ago and no money yet", "billing"),
    ("Necesito factura con RUC para mi empresa", "billing"), ("why is there a $4.99 fee on my statement from you", "billing"),
    ("my card got declined but the money left my account", "billing"), ("¿Aceptan Yape o Plin?", "billing"),
    ("my package says delivered but it's not here", "shipping"), ("¿cuándo llega mi pedido a Arequipa?", "shipping"),
    ("tracking number doesnt work", "shipping"), ("can I change the delivery address, it hasn't shipped yet", "shipping"),
    ("Order placed 10 days ago still 'processing'", "shipping"), ("el courier no vino hoy como dijeron", "shipping"),
    ("Do you ship to Bolivia?", "shipping"), ("Estimated delivery passed yesterday. Any update?", "shipping"),
    ("the jacket is too small, can I exchange it for an L", "returns"), ("me llegó el producto roto", "returns"),
    ("you sent me a blue one, I ordered black", "returns"), ("how do I print the return label", "returns"),
    ("quiero devolver la licuadora, no me gustó", "returns"), ("Is there a time limit to return electronics?", "returns"),
    ("box arrived crushed and the glass is cracked", "returns"), ("Puedo cambiar las zapatillas por otro color?", "returns"),
    ("I can't log in, it says wrong password but I reset it", "account"), ("cómo cambio el correo de mi cuenta", "account"),
    ("please delete all my data and my account", "account"), ("never got the verification email", "account"),
    ("someone else is using my account I think", "account"), ("no puedo entrar con Google", "account"),
    ("How do I turn off marketing emails?", "account"), ("update my phone number on profile", "account"),
    ("does this laptop have a backlit keyboard?", "product"), ("¿la talla M equivale a qué medida de pecho?", "product"),
    ("Is the XR-200 compatible with iPhone 15?", "product"), ("when will the green one be back in stock", "product"),
    ("¿Es resistente al agua?", "product"), ("what's the difference between the pro and the regular", "product"),
    ("does the coffee maker come with a filter", "product"), ("tienen este modelo en 220V?", "product"),
    ("I'd like to sell my brand on your marketplace", "other"), ("your app is great, thanks!", "other"),
    ("are you hiring in Lima?", "other"), ("hola", "other"), ("CLICK HERE to win a free iPhone!!!", "other"),
    ("Quisiera proponer una alianza con mi tienda", "other"), ("what are your store opening hours on sunday", "other"),
    ("I have a complaint about one of your delivery drivers' behaviour", "other"),
]
print(len(GOLD), "gold examples")

## 2. The teacher, zero-shot
Before distilling, know the teacher's own accuracy: the student can't learn what the teacher gets wrong.
Batch labelling (12 messages per call) is cheaper, but check the number of labels returned — and never fill failures with a default label.

In [ ]:
def parse_json(text):
    m = re.search(r"[\[{].*[\]}]", text or "", re.S)
    try:
        return json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        return None


def teacher_label(messages):
    """Label a batch of messages in one call (cheaper than one call each). Returns a list of intents (None = teacher failed)."""
    batch = json.dumps([{"id": i, "text": m} for i, m in enumerate(messages)], ensure_ascii=False)  # JSON: robust to newlines in messages
    prompt = (f"Classify each customer message into exactly one queue.\nQueues:\n"
              + "\n".join(f"- {k}: {v}" for k, v in INTENTS.items())
              + f'\n\nMessages (JSON):\n{batch}\n\nReturn JSON mapping every id to its queue: {{"labels": {{"0": "<queue>", "1": "<queue>", ...}}}}')
    for attempt in range(5):
        try:
            r = tc.chat.completions.create(model=TMODEL, messages=[{"role": "user", "content": prompt}], temperature=0, max_tokens=4000, **EXTRA)
            labels = (parse_json(r.choices[0].message.content) or {}).get("labels", {})
            if isinstance(labels, dict) and all(str(i) in labels for i in range(len(messages))):
                return [labels[str(i)] if labels[str(i)] in INTENTS else None for i in range(len(messages))]
            print(f"teacher returned {len(labels)} labels for {len(messages)} messages, retrying")
        except Exception as e:
            print("teacher error:", type(e).__name__); time.sleep(15)
    print(f"⚠️ teacher failed on a batch of {len(messages)}")  # fail loudly: a silent default label would poison the data
    return [None] * len(messages)


start = time.perf_counter()
teacher_gold = sum((teacher_label([m for m, _ in GOLD[i: i + 12]]) for i in range(0, len(GOLD), 12)), [])
teacher_secs = (time.perf_counter() - start) / 4
assert None not in teacher_gold, "teacher failed on part of the gold set: re-run this cell"
y_gold = [l for _, l in GOLD]
print(f"teacher accuracy on gold: {accuracy_score(y_gold, teacher_gold):.0%}  ({teacher_secs:.1f}s per batch call)")

## 3. Generate: diversity comes from the seeds, not from "be diverse"
Asking an LLM "give me 300 varied examples" yields 300 near-identical ones. Instead, enumerate **seed attributes** (intent × language ×
persona × style) and ask for a few examples per combination — the same trick as 10.02's synthetic eval sets.

In [ ]:
LANGS = ["English", "Spanish (Peru)", "Spanglish / mixed"]
PERSONAS = ["an angry customer", "a polite elderly person", "a busy professional typing on a phone", "a teenager", "a small business owner"]
STYLES = ["very short, no punctuation", "one or two sentences", "detailed, with an order number", "with typos"]

rng = random.Random(0)
combos = list(itertools.product(INTENTS, LANGS, PERSONAS, STYLES))
rng.shuffle(combos)
synth_path = DATA / "distill-synth-raw.json"
raw = json.loads(synth_path.read_text()) if synth_path.exists() else []
start = time.perf_counter()
for intent, lang, persona, style in combos[: 60 - len(raw) // 5]:
    prompt = (f"Write 5 different messages that {persona} might send to an online store's support chat, in {lang}, style: {style}. "
              f"Topic: {intent} ({INTENTS[intent]}). Vary the specific situation in each. Return JSON: {{\"messages\": [\"...\", ...]}}")
    try:
        r = tc.chat.completions.create(model=TMODEL, messages=[{"role": "user", "content": prompt}], temperature=1.0, max_tokens=3000, **EXTRA)
        msgs = (parse_json(r.choices[0].message.content) or {}).get("messages", [])
        raw += [{"text": m, "intent": intent, "lang": lang} for m in msgs if isinstance(m, str) and m.strip()]
    except Exception as e:
        print("gen error:", type(e).__name__); time.sleep(15)
    time.sleep(PAUSE)
synth_path.write_text(json.dumps(raw, ensure_ascii=False, indent=1))
print(f"{len(raw)} raw examples ({time.perf_counter() - start:.0f}s)")
for e in rng.sample(raw, 5):
    print(f"  [{e['intent']:8}] {e['text'][:90]}")

## 4. Clean: deduplicate and filter
- **Near-duplicates** (cosine similarity > 0.9) add no information and over-weight the teacher's favourite phrasings.
- **Label filter**: the generator was *asked* for an intent, but did it write one? Re-label every example with the teacher and keep only those
  where request and label agree (a cheap verifier; rejection sampling in spirit).

In [ ]:
emb = rag.embed([e["text"] for e in raw])
keep, kept_emb = [], []
for e, v in zip(raw, emb):
    if not kept_emb or (np.stack(kept_emb) @ v).max() < 0.9:
        keep.append(e); kept_emb.append(v)
print(f"dedupe: {len(raw)} -> {len(keep)}")

relabel_path = DATA / "distill-relabel.json"
relabels = json.loads(relabel_path.read_text()) if relabel_path.exists() and len(json.loads(relabel_path.read_text())) == len(keep) else None
if relabels is None:
    relabels = []
    for i in range(0, len(keep), 20):
        relabels += teacher_label([e["text"] for e in keep[i: i + 20]]); time.sleep(PAUSE)
    relabel_path.write_text(json.dumps(relabels))
clean = [e for e, l in zip(keep, relabels) if l == e["intent"]]
print(f"label filter: {len(keep)} -> {len(clean)} ({1 - len(clean) / len(keep):.0%} disagreed or failed; {relabels.count(None)} teacher failures)")
for e, l in [(e, l) for e, l in zip(keep, relabels) if l != e["intent"]][:4]:
    print(f"  asked {e['intent']:8} -> teacher says {l:8}: {e['text'][:70]}")

**Diversity check**: distinct-n (fraction of unique n-grams) and how spread out the embeddings are. Compare against the gold set:
synthetic data that's much *less* diverse than real data is the most common reason distilled students fail in production.

In [ ]:
def distinct_n(texts, n=2):
    grams = [tuple(t.lower().split()[i: i + n]) for t in texts for i in range(len(t.split()) - n + 1)]
    return len(set(grams)) / max(len(grams), 1)


def spread(texts):
    v = rag.embed(texts); sims = v @ v.T
    return 1 - sims[np.triu_indices(len(v), 1)].mean()


sample = rng.sample(clean, min(len(GOLD), len(clean)))
print(f"distinct-2: synthetic {distinct_n([e['text'] for e in sample]):.2f} vs gold {distinct_n([m for m, _ in GOLD]):.2f}")
print(f"embedding spread: synthetic {spread([e['text'] for e in sample]):.3f} vs gold {spread([m for m, _ in GOLD]):.3f}")
print("avg length (words): synthetic", round(np.mean([len(e["text"].split()) for e in clean]), 1), "| gold", round(np.mean([len(m.split()) for m, _ in GOLD]), 1))

## 5. Train the student
Embeddings from 07.x (`bge-small`) + logistic regression. Training takes milliseconds; that's the point.

In [ ]:
def train_student(examples):
    X = rag.embed([e["text"] for e in examples])
    return LogisticRegression(max_iter=2000, C=4.0).fit(X, [e["intent"] for e in examples])


X_gold = rag.embed([m for m, _ in GOLD])
student = train_student(clean)
t = time.perf_counter(); pred = student.predict(rag.embed([m for m, _ in GOLD])); student_ms = (time.perf_counter() - t) / len(GOLD) * 1000
print(f"student accuracy on gold: {accuracy_score(y_gold, pred):.0%} ({student_ms:.1f} ms/message incl. embedding)")
print(f"teacher accuracy on gold: {accuracy_score(y_gold, teacher_gold):.0%} (~{teacher_secs * 1000 / 12:.0f} ms/message batched, {teacher_secs * 1000:.0f} ms per call)")
print(classification_report(y_gold, pred, zero_division=0))

## 6. What mattered? Ablations
Same student, different training data: raw vs deduplicated vs filtered, and how many examples you need.

In [ ]:
for name, data in [("raw (no cleaning)", raw), ("deduplicated", keep), ("dedup + label filter", clean)]:
    print(f"{name:22} {len(data):4} examples -> gold accuracy {accuracy_score(y_gold, train_student(data).predict(X_gold)):.0%}")
for n in (30, 60, 120, len(clean)):
    accs = [accuracy_score(y_gold, train_student(random.Random(s).sample(clean, min(n, len(clean)))).predict(X_gold)) for s in range(3)]
    print(f"{min(n, len(clean)):4} clean examples -> {np.mean(accs):.0%} ± {np.std(accs):.0%}")

Read these numbers with care: on 48 test examples one message = 2 points, so differences of a few points between the cleaning variants are
noise (filtering mattered a lot when the teacher batch-labelling was buggy; with clean labels it removes only a few examples). The learning
curve is the informative part: accuracy is still rising, but slowly.

The big signal is the **gap to the teacher**. Look back at §4's diagnostics: synthetic messages are ~4× longer and less diverse than real
ones. Real users type "tracking number doesnt work"; the generator writes polite paragraphs with order numbers. The student learned the
generator's style. Fix the *distribution*, not the model:

## 7. Fix the mismatch: seeds that look like real traffic

In [ ]:
SHORT_STYLES = ["3 to 8 words, chat style, no greeting", "one short sentence, at most 12 words, lowercase", "a fragment like a search query"]
short_combos = list(itertools.product(INTENTS, LANGS, PERSONAS[:3], SHORT_STYLES))
random.Random(1).shuffle(short_combos)
short_path = DATA / "distill-synth-short.json"
short_raw = json.loads(short_path.read_text()) if short_path.exists() else []
for intent, lang, persona, style in short_combos[: 36 - len(short_raw) // 5]:
    prompt = (f"Write 5 different messages that {persona} might send to an online store's support chat, in {lang}, style: {style}. "
              f"Topic: {intent} ({INTENTS[intent]}). Vary the situation. Return JSON: {{\"messages\": [\"...\", ...]}}")
    try:
        r = tc.chat.completions.create(model=TMODEL, messages=[{"role": "user", "content": prompt}], temperature=1.0, max_tokens=3000, **EXTRA)
        short_raw += [{"text": m, "intent": intent, "lang": lang} for m in (parse_json(r.choices[0].message.content) or {}).get("messages", [])
                      if isinstance(m, str) and m.strip()]
    except Exception as e:
        print("gen error:", type(e).__name__); time.sleep(15)
    time.sleep(PAUSE)
short_path.write_text(json.dumps(short_raw, ensure_ascii=False, indent=1))
short_labels = sum((teacher_label([e["text"] for e in short_raw[i: i + 20]]) for i in range(0, len(short_raw), 20)), [])
short_clean = [e for e, l in zip(short_raw, short_labels) if l == e["intent"]]
print(f"{len(short_raw)} short examples -> {len(short_clean)} after label filter | avg {np.mean([len(e['text'].split()) for e in short_clean]):.1f} words")

for name, data in [("long only (before)", clean), ("short only", short_clean), ("long + short", clean + short_clean)]:
    print(f"{name:20} {len(data):4} examples -> student gold accuracy {accuracy_score(y_gold, train_student(data).predict(X_gold)):.0%}")
print(f"teacher: {accuracy_score(y_gold, teacher_gold):.0%}")

An honest result: short-only data is *worse* (fewer examples, narrower coverage), and long + short gains only a few points — a couple of
messages on this test set. So length mismatch was real but **not the main bottleneck**. Check the classification report in §5: the errors
concentrate on `other` (a grab-bag class with no shared vocabulary) and on boundary cases ("where is my refund" — billing or returns?).
That's the **student's capacity**: a linear model on sentence embeddings matches topics, not intent nuances that a 120B model reads.

What to do next, in order of cost: make the label definitions sharper (the teacher and the student both benefit), add seeds for the confused
boundaries, use a stronger student (fine-tuned small LLM, exercise 2), or keep the teacher for the hard cases (confidence cascade, exercise 3).
And keep the rule from this section: **look at real traffic first**, design seeds to cover it, and always measure on human-labelled data.

## 8. Distillation variants
- **Hard-label** (what we did): teacher outputs a label/answer, student trains on it. Works with any API.
- **Soft-label / logit distillation**: student matches the teacher's full probability distribution, `KL(softmax(z_t / T) ‖ softmax(z_s / T))`.
  Much richer signal ("this is 70% billing, 25% returns") but needs the teacher's logits (open weights, same tokenizer) — how Gemma and
  Llama 3.2 small models were trained.
- **Sequence-level / SFT distillation**: fine-tune a small LLM on the teacher's full responses (12.03 with teacher-written targets); DeepSeek-R1's
  distilled Qwen/Llama models came from 800k teacher reasoning traces.
- **On-policy distillation**: the student generates, the teacher scores/corrects the student's own outputs (fixes the student's actual mistakes).
- **Rejection sampling / self-distillation**: generate many candidates, keep those a verifier accepts (unit tests, answer checks — 12.05, 15.01).

## 9. Licences and terms of service
Model outputs aren't automatically yours to train on:
- **OpenAI, Anthropic, Google (Gemini)** terms forbid using outputs to develop models that **compete** with them; fine-tuning for your own narrow
  task is generally allowed, but read the current terms for your use case.
- **Open-weight licences** differ: Apache-2.0 / MIT models (Qwen, gpt-oss, DeepSeek, Mistral's Apache models) put no restrictions on outputs;
  **Llama**'s licence requires derived models to include "Llama" in the name and to follow its acceptable-use policy; Gemma has its own terms.
- **Hosted open models** (Groq, Together, OpenRouter) add the host's terms on top of the model's licence.
- Record **provenance**: which model, prompt and date produced each example (a column in your dataset). You'll need it for audits and for
  regenerating data when a better teacher arrives.

## Exercises
1. Distill a frontier model's behaviour on **one task of yours** into your small model: generate 500 examples from seeds, clean them, train a
   student, and evaluate on 50 hand-labelled real examples.
2. Replace the logistic regression with a fine-tuned `SmolLM2-135M` (12.03 LoRA recipe) that outputs the label. Is it better than the
   embedding student? At what latency?
3. Add a **confidence cascade**: route messages where the student's top probability < 0.6 to the teacher (13.06). Plot accuracy vs % sent to the teacher.
4. Implement soft-label distillation with an open teacher: get class probabilities from `qwen3:1.7b` by scoring each label's log-likelihood,
   then train the student on those soft targets. Compare with hard labels.

In [ ]:
# your code here